# Writing Group A's adapter

Group A keeps the Swiss system in four semicolon-separated CSV files with German headers. Technology is a number. The reservoir is a second table, and which row belongs to which plant is only a convention.

An adapter is the only place those conventions are written down. It has two functions:

- `to_cesdm` reads the group's files and writes CESDM objects.
- `from_cesdm` reads CESDM objects and writes the group's files.

This notebook builds that adapter in the same order as `docs/examples/four-research-groups/adapter_a.py`. The finished module stays the one the exchange uses. The last cell checks that these cells describe the same system.


## Setup

Find the repository, open Group A's tables, and import the two calls the adapter needs: `build_model_from_yaml` to open a model, and `get_attr_value` to read one back.


In [1]:
from pathlib import Path
import csv
import os
import sys

here = Path.cwd().resolve()
REPO = next(
    (p for p in (here, *here.parents) if (p / "cesdm" / "schemas" / "base").is_dir()),
    None,
)
if REPO is None:
    raise FileNotFoundError(
        "Could not find cesdm/schemas/base. Open this notebook from the cesdm-toolbox checkout."
    )

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from cesdm_toolbox import build_model_from_yaml
from ear import get_attr_value

DATA = REPO / "docs" / "examples" / "four-research-groups" / "group-a-csv"
SOURCE = "group-a"

for name in ("regions.csv", "load.csv", "plants.csv", "storage.csv"):
    print(name)
    print((DATA / name).read_text(encoding="utf-8").rstrip())
    print()


regions.csv
ID;Bezeichnung
CH;Schweiz

load.csv
Region;E_Jahr
CH;58000

plants.csv
ID;Region;Pmax;Typ;Eta;VarKosten
CH_GAS;CH;400;1;58;45
CH_NUC;CH;3000;2;33;9
CH_WIND;CH;100;3;;0
CH_PV;CH;6500;4;;0
CH_HYD_RES;CH;8500;6;90;0
CH_HYD_ROR;CH;4000;7;90;0

storage.csv
ID;Region;E_speicher;Zufluss
CH_RES;CH;8800000;20000000
CH_ROR;CH;0;16000000



## 1. Write down the mapping

Before creating any entity, decide what each column means in CESDM. A column is either a value with a unit, or a link to another object.

| Group A | Meaning | In CESDM |
|---|---|---|
| a row in `plants.csv` | a power plant | a generation entity |
| `Region = CH` | located in Switzerland | link `belongsToGeographicalRegion` → `region.country.CH` |
| `Typ` | which technology | attribute `generator_technology_type` |
| `Eta` | conversion efficiency, percent | `energy_conversion_efficiency`, fraction (÷ 100) |
| `VarKosten` | variable operating cost, EUR/MWh | `variable_operating_cost`, MU/MWh |
| `Pmax` | installed capacity, MW | `nominal_power_capacity`, MW |
| a row in `storage.csv` | energy and inflow of a hydro plant | a `HydroReservoir`, linked from that plant |
| `E_Jahr` | annual demand, GWh/year | `DemandUnit.annual_energy_demand`, MWh/year (× 1000) |

`Typ` is Group A's own code. It does not point at a library technology. Efficiency and variable cost stay on the plant, because they come from the row.


In [2]:
REGIONS = {"CH": ("region.country.CH", "Schweiz")}

TYP = {
    1: "ccgt",
    2: "nuclear",
    3: "wind",
    4: "solar",
    6: "reservoir_hydro",
    7: "run_of_river",
}
RESERVOIR, RUN_OF_RIVER = 6, 7
TECH_TO_TYP = {tech: typ for typ, tech in TYP.items()}
REGION_TO_CODE = {cesdm: code for code, (cesdm, _) in REGIONS.items()}

KLASSE = {
    "wind": "WindGenerationUnit",
    "solar": "SolarGenerationUnit",
}
GENERATOR_CLASSES = (
    "GenerationUnit",
    "WindGenerationUnit",
    "SolarGenerationUnit",
    "HydroGenerationUnit",
)

for code, name in TYP.items():
    print(f"Typ {code} → {name}")


Typ 1 → ccgt
Typ 2 → nuclear
Typ 3 → wind
Typ 4 → solar
Typ 6 → reservoir_hydro
Typ 7 → run_of_river


## 2. Open an empty model

Group A opens the schema and the libraries itself. The region code `CH` will point at `region.country.CH` from the regions library. Nothing from the CSV is in the model yet.


In [3]:
def new_model():
    model = build_model_from_yaml(str(REPO / "cesdm" / "schemas" / "base"))
    model.import_library(str(REPO / "cesdm" / "library" / "default_library"))
    model.import_library(str(REPO / "cesdm" / "library" / "regions_library"))
    return model

def read_csv(path):
    with path.open(encoding="utf-8") as handle:
        return list(csv.DictReader(handle, delimiter=";"))

model = new_model()
print("schema loaded, group tables not imported yet")
print("regions available:", "region.country.CH" in model.get_entities("GeographicalRegion"))


schema loaded, group tables not imported yet
regions available: True


## 3. Demand

One row of `load.csv` becomes one `DemandUnit`, one region link, and one annual demand. `E_Jahr` is in GWh/year. CESDM stores `annual_energy_demand` in MWh/year, so the adapter multiplies by 1000.

`unit=` is checked when the attribute is written. A value in GWh/year is rejected here, in Group A's code, before a file is sent.


In [4]:
scratch = new_model()
scratch.add_entity("DemandUnit", "demand.try")
try:
    scratch.add_attribute(
        "demand.try", "annual_energy_demand", 58000, unit="GWh/year", provenance_ref=SOURCE
    )
except ValueError as exc:
    print("rejected:", exc)

for row in read_csv(DATA / "load.csv"):
    region = REGIONS[row["Region"]][0]
    eid = f"demand.{row['Region']}"
    model.add_entity("DemandUnit", eid)
    model.add_relation(eid, "belongsToGeographicalRegion", region)
    model.add_attribute(
        eid, "annual_energy_demand", float(row["E_Jahr"]) * 1000,
        unit="MWh/year", provenance_ref=SOURCE,
    )
    demand = model.get_entity(eid)
    print(eid, demand.get_attr_value("annual_energy_demand"), "MWh/year")
    print("  belongsToGeographicalRegion →", model.get_entity(eid).get_relation("belongsToGeographicalRegion"))


rejected: [DemandUnit:demand.try] Unit 'GWh/year' is not allowed for 'annual_energy_demand'. Allowed: ['MWh/year']
demand.CH 58000000.0 MWh/year
  belongsToGeographicalRegion → region.country.CH


## 4. Plants

Each row of `plants.csv` is one plant. Wind and photovoltaics have their own classes. Reservoir and run-of-river are both a `HydroGenerationUnit`. Gas and nuclear stay a generic `GenerationUnit`.

The category is the string from the `TYP` table, stored as `generator_technology_type`. Capacity stays in MW. Cost stays a number; the unit becomes `MU/MWh`.

Wind and photovoltaics have an empty `Eta`. The attribute is left unset. An empty cell is not written as zero, and it is not filled with a default efficiency.


In [5]:
for row in read_csv(DATA / "plants.csv"):
    typ = int(row["Typ"])
    art = TYP[typ]
    region = REGIONS[row["Region"]][0]
    eid = f"gen.{row['ID']}"
    if typ in (RESERVOIR, RUN_OF_RIVER):
        cesdm_class = "HydroGenerationUnit"
    else:
        cesdm_class = KLASSE.get(art, "GenerationUnit")
    model.add_entity(cesdm_class, eid)
    model.add_relation(eid, "belongsToGeographicalRegion", region)
    model.add_attribute(eid, "nominal_power_capacity", float(row["Pmax"]), unit="MW", provenance_ref=SOURCE)
    model.add_attribute(eid, "generator_technology_type", art)
    if str(row["Eta"]).strip():
        model.add_attribute(
            eid, "energy_conversion_efficiency", float(row["Eta"]) / 100,
            unit="fraction", provenance_ref=SOURCE,
        )
    model.add_attribute(
        eid, "variable_operating_cost", float(row["VarKosten"]),
        unit="MU/MWh", provenance_ref=SOURCE,
    )
    plant = model.get_entity(eid)
    eta = plant.get_attr_value("energy_conversion_efficiency")
    print(
        f"{eid:16} {cesdm_class:24} {art:16} "
        f"{plant.get_attr_value('nominal_power_capacity')} MW  "
        f"eta {eta if eta is not None else '—'}  "
        f"cost {plant.get_attr_value('variable_operating_cost')}"
    )


gen.CH_GAS       GenerationUnit           ccgt             400.0 MW  eta 0.58  cost 45.0
gen.CH_NUC       GenerationUnit           nuclear          3000.0 MW  eta 0.33  cost 9.0
gen.CH_WIND      WindGenerationUnit       wind             100.0 MW  eta —  cost 0.0
gen.CH_PV        SolarGenerationUnit      solar            6500.0 MW  eta —  cost 0.0
gen.CH_HYD_RES   HydroGenerationUnit      reservoir_hydro  8500.0 MW  eta 0.9  cost 0.0
gen.CH_HYD_ROR   HydroGenerationUnit      run_of_river     4000.0 MW  eta 0.9  cost 0.0


## 5. Make the reservoir link explicit

`storage.csv` does not name a plant. Group A's convention is: in the same region, the row with stored energy belongs to `Typ` 6, and the row with no stored energy belongs to `Typ` 7.

The adapter resolves that convention once. Each hydro plant then draws from its own `HydroReservoir` through `drawsFromHydroReservoir`. A receiver never has to guess which row belonged to which plant.


In [6]:
storage = {}
for row in read_csv(DATA / "storage.csv"):
    kind = RESERVOIR if float(row["E_speicher"]) > 0 else RUN_OF_RIVER
    storage[(row["Region"], kind)] = row
    print(f"{row['ID']:8} region {row['Region']}  E_speicher {row['E_speicher']:>8}  → Typ {kind} ({TYP[kind]})")

print()
for row in read_csv(DATA / "plants.csv"):
    typ = int(row["Typ"])
    if typ not in (RESERVOIR, RUN_OF_RIVER):
        continue
    basin_row = storage[(row["Region"], typ)]
    eid = f"gen.{row['ID']}"
    basin = f"storage.{basin_row['ID']}"
    model.add_entity("HydroReservoir", basin)
    model.add_attribute(basin, "energy_storage_capacity", float(basin_row["E_speicher"]), unit="MWh", provenance_ref=SOURCE)
    model.add_attribute(basin, "annual_natural_inflow_energy", float(basin_row["Zufluss"]), unit="MWh/year", provenance_ref=SOURCE)
    model.add_relation(eid, "drawsFromHydroReservoir", basin)
    print(f"{eid} drawsFromHydroReservoir → {basin}")
    store = model.get_entity(basin)
    print(f"  energy {store.get_attr_value('energy_storage_capacity')} MWh")
    print(f"  inflow {store.get_attr_value('annual_natural_inflow_energy')} MWh/year")


CH_RES   region CH  E_speicher  8800000  → Typ 6 (reservoir_hydro)
CH_ROR   region CH  E_speicher        0  → Typ 7 (run_of_river)

gen.CH_HYD_RES drawsFromHydroReservoir → storage.CH_RES
  energy 8800000.0 MWh
  inflow 20000000.0 MWh/year
gen.CH_HYD_ROR drawsFromHydroReservoir → storage.CH_ROR
  energy 0.0 MWh
  inflow 16000000.0 MWh/year


## 6. Check the file before sending it

`validate()` checks the model against the schema. An empty list means the file is valid CESDM. The YAML below is what another group would load: demand in MWh/year, nuclear as a category plus its own efficiency, wind with no efficiency attribute, and the reservoir linked to its basin.


In [7]:
errors = model.validate()
print(f"{len(errors)} schema errors")
for line in errors:
    print(" ", line)

model.export_yaml(REPO / "output" / "adapter-a-notebook" / "group_a.yaml")
text = (REPO / "output" / "adapter-a-notebook" / "group_a.yaml").read_text(encoding="utf-8").splitlines()

def excerpt(marker):
    start = next(i for i, line in enumerate(text) if line.startswith(marker))
    end = start + 1
    while end < len(text):
        line = text[end]
        if line.endswith(":") and (not line.startswith(" ") or (line.startswith("  ") and not line.startswith("   "))):
            break
        end += 1
    print()
    print("\n".join(text[start:end]))

excerpt("  demand.CH:")
excerpt("  gen.CH_NUC:")
excerpt("  gen.CH_WIND:")
excerpt("  gen.CH_HYD_RES:")
excerpt("  storage.CH_RES:")


0 schema errors

  demand.CH:
    attributes:
    - id: annual_energy_demand
      value: 58000000.0
      unit: MWh/year
      provenance_ref: group-a
    relations:
    - id: hasCarrier
      target_entity_ids:
      - carrier.electricity
    - id: belongsToGeographicalRegion
      target_entity_ids:
      - region.country.CH

  gen.CH_NUC:
    attributes:
    - id: generator_technology_type
      value: nuclear
    - id: nominal_power_capacity
      value: 3000.0
      unit: MW
      provenance_ref: group-a
    - id: variable_operating_cost
      value: 9.0
      unit: MU/MWh
      provenance_ref: group-a
    - id: energy_conversion_efficiency
      value: 0.33
      unit: fraction
      provenance_ref: group-a
    relations:
    - id: belongsToGeographicalRegion
      target_entity_ids:
      - region.country.CH

  gen.CH_WIND:
    attributes:
    - id: generator_technology_type
      value: wind
    - id: nominal_power_capacity
      value: 100.0
      unit: MW
      provenance_re

## 7. Write Group A's files again

`from_cesdm` reads the same CESDM objects every sender produces. It puts technologies back into Group A's codes, demand back into GWh/year, and efficiency back into percent. A missing efficiency is an empty `Eta` cell.

Storage and inflow leave the plant relation and go back into `storage.csv`. The identifiers in the file are the ones on the CESDM objects. Here they are still Group A's, because this model was built from Group A's tables.


In [8]:
def zahl(x):
    x = round(float(x), 9)
    return int(x) if x.is_integer() else x

def write_csv(path, header, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle, delimiter=";", lineterminator="\n")
        writer.writerow(header)
        writer.writerows(rows)

basins = model.get_entities("HydroReservoir")
plants = []
for cesdm_class in GENERATOR_CLASSES:
    for eid, gen in model.get_entities(cesdm_class).items():
        record = {
            "id": eid.removeprefix("gen."),
            "region": gen.get_relation("belongsToGeographicalRegion"),
            "technology": get_attr_value(gen, "generator_technology_type"),
            "capacity_mw": get_attr_value(gen, "nominal_power_capacity"),
            "efficiency": get_attr_value(gen, "energy_conversion_efficiency"),
            "variable_cost": get_attr_value(gen, "variable_operating_cost"),
        }
        sid = gen.get_relation("drawsFromHydroReservoir")
        if sid:
            basin = basins[sid]
            record.update(
                storage_id=sid.removeprefix("storage."),
                storage_mwh=get_attr_value(basin, "energy_storage_capacity"),
                inflow_mwh_per_year=get_attr_value(basin, "annual_natural_inflow_energy"),
            )
        plants.append(record)

demand = [
    {
        "region": unit.get_relation("belongsToGeographicalRegion"),
        "annual_mwh": get_attr_value(unit, "annual_energy_demand"),
    }
    for unit in model.get_entities("DemandUnit").values()
]

out = REPO / "output" / "adapter-a-notebook" / "csv"
codes = sorted({REGION_TO_CODE[p["region"]] for p in plants})
write_csv(out / "regions.csv", ["ID", "Bezeichnung"], [[c, REGIONS[c][1]] for c in codes])
write_csv(out / "load.csv", ["Region", "E_Jahr"], [[REGION_TO_CODE[d["region"]], zahl(d["annual_mwh"] / 1000)] for d in demand])
write_csv(
    out / "plants.csv",
    ["ID", "Region", "Pmax", "Typ", "Eta", "VarKosten"],
    [[
        p["id"], REGION_TO_CODE[p["region"]], zahl(p["capacity_mw"]), TECH_TO_TYP[p["technology"]],
        "" if p["efficiency"] is None else zahl(p["efficiency"] * 100), zahl(p["variable_cost"]),
    ] for p in plants],
)
write_csv(
    out / "storage.csv",
    ["ID", "Region", "E_speicher", "Zufluss"],
    [[p["storage_id"], REGION_TO_CODE[p["region"]], zahl(p["storage_mwh"]), zahl(p["inflow_mwh_per_year"])]
     for p in plants if "storage_id" in p],
)

for name in ("load.csv", "plants.csv", "storage.csv"):
    print(name)
    print((out / name).read_text(encoding="utf-8").rstrip())
    print()


load.csv
Region;E_Jahr
CH;58000

plants.csv
ID;Region;Pmax;Typ;Eta;VarKosten
CH_GAS;CH;400;1;58;45
CH_NUC;CH;3000;2;33;9
CH_WIND;CH;100;3;;0
CH_PV;CH;6500;4;;0
CH_HYD_RES;CH;8500;6;90;0
CH_HYD_ROR;CH;4000;7;90;0

storage.csv
ID;Region;E_speicher;Zufluss
CH_RES;CH;8800000;20000000
CH_ROR;CH;0;16000000



## 8. The same system as `adapter_a.py`

Read the CSV just written back into a new model, and build a second model with the finished `adapter_a.py` from the original tables. Both are compared on region and technology category: capacity, efficiency, cost, storage and inflow. Identifiers are not part of that comparison.

Zero differences means this notebook and `adapter_a.py` carry the same system, and the trip back through Group A's columns did not drop the empty wind and photovoltaics efficiencies.


In [9]:
groups = REPO / "docs" / "examples" / "four-research-groups"
if str(groups) not in sys.path:
    sys.path.insert(0, str(groups))

import adapter_a
from compare import compare

returned = adapter_a.leeres_modell()
adapter_a.to_cesdm(out, returned)
official = adapter_a.leeres_modell()
adapter_a.to_cesdm(DATA, official)

for label, other in (("written CSV", returned), ("adapter_a.py", official)):
    diffs = compare(model, other, "notebook", label)
    print(f"notebook vs {label}: {len(diffs)} differences")
    for line in diffs:
        print(" ", line)


notebook vs written CSV: 0 differences
notebook vs adapter_a.py: 0 differences
